# AML 3303 — Week 3 In-Class Lab (30 minutes)
### pandas + NumPy only

## Scenario

**NorthLine Electronics** is an online store. The ops team exported two tables for you:
`customers` and `orders`. They want a quick read on which product categories actually
make money, and which city orders the most.

The export is realistic, which means it is slightly messy: a missing quantity, a missing
price, a duplicate order, and one order from a customer who is not in the customer list.

Fill in every cell marked `# TODO`. The hints sit directly above each one.

## 0. Setup

Run this cell once. Do not change it.

In [148]:
# Run this cell first. It creates the two DataFrames you will work with.
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Aarav Shah","Bianca Rossi","Chen Wei","Diya Patel",
             "Emeka Obi","Farah Haddad","Grace Kim","Hugo Martin"],
    "city": ["Ottawa","Toronto","Ottawa","Montreal",
             "Toronto","Calgary","Montreal","Ottawa"],
})

orders = pd.DataFrame({
    "order_id":   [101,102,103,104,105,106,107,108,109,110,
                   111,112,113,114,115,116,117,118,119,120,118,121],
    "customer_id":[  1,  2,  1,  3,  4,  5,  2,  6,  3,  7,
                     8,  5,  1,  4,  6,  7,  2,  8,  3,  5,  8,  9],
    "order_date": ["2024-05-02","2024-05-02","2024-05-04","2024-05-05","2024-05-07",
                   "2024-05-08","2024-05-09","2024-05-11","2024-05-12","2024-05-13",
                   "2024-05-15","2024-05-16","2024-05-18","2024-05-19","2024-05-20",
                   "2024-05-22","2024-05-23","2024-05-25","2024-05-26","2024-05-28",
                   "2024-05-25","2024-05-29"],
    "category":   ["Laptops","Headphones","Headphones","Monitors","Laptops",
                   "Keyboards","Monitors","Laptops","Keyboards","Headphones",
                   "Monitors","Laptops","Keyboards","Headphones","Monitors",
                   "Laptops","Keyboards","Headphones","Laptops","Monitors",
                   "Headphones","Keyboards"],
    "quantity":   [1,2,1,2,1,3,1,1,np.nan,1,4,1,2,1,1,1,1,2,1,2,2,1],
    "unit_price": [1299.99,89.50,89.50,219.00,1499.00,45.00,219.00,1099.00,45.00,np.nan,
                   219.00,1299.99,45.00,89.50,219.00,1499.00,45.00,89.50,1299.99,219.00,
                   89.50,45.00],
    "status":     ["delivered","delivered","returned","delivered","cancelled",
                   "delivered","delivered","delivered","delivered","delivered",
                   "delivered","returned","delivered","delivered","delivered",
                   "delivered","cancelled","delivered","delivered","delivered",
                   "delivered","delivered"],
})

print("orders:", orders.shape, " customers:", customers.shape)
orders.head()

orders: (22, 7)  customers: (8, 3)


,order_id,customer_id,order_date,category,quantity,unit_price,status
0,101,1,2024-05-02,Laptops,1.0,1299.99,delivered
1,102,2,2024-05-02,Headphones,2.0,89.50,delivered
2,103,1,2024-05-04,Headphones,1.0,89.50,returned
3,104,3,2024-05-05,Monitors,2.0,219.00,delivered
4,105,4,2024-05-07,Laptops,1.0,1499.00,cancelled


## 1. First look (3 min)

**Hints**
- `df.shape` → rows and columns
- `df.info()` → column types and non-null counts
- `df.describe()` → quick numeric summary
- `df["category"].value_counts()` → how many orders per category

In [149]:
# TODO: print the shape, run info(), and show how many orders there are per category
customers.shape
customers.info()
customers.describe()
print('\n')
print('Customers per city',customers['city'].value_counts())
print('\n')
orders.shape
orders.info()
orders.describe()
print('\n')
print('Orders per category',orders['category'].value_counts())


<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  8 non-null      int64
 1   name         8 non-null      str  
 2   city         8 non-null      str  
dtypes: int64(1), str(2)
memory usage: 324.0 bytes


Customers per city city
Ottawa      3
Toronto     2
Montreal    2
Calgary     1
Name: count, dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     22 non-null     int64  
 1   customer_id  22 non-null     int64  
 2   order_date   22 non-null     str    
 3   category     22 non-null     str    
 4   quantity     21 non-null     float64
 5   unit_price   21 non-null     float64
 6   status       22 non-null     str    
dtypes: float64(2), int64(2), str(3)
memory usage: 1.3 KB


Orders per category c

## 2. A first (wrong) answer (5 min)

Compute revenue per category **before** cleaning anything, counting delivered orders only.
Keep this result: you will compare against it at the end.

**Hints**
- Filter rows: `orders[orders["status"] == "delivered"]`
- New column: `df["revenue"] = df["quantity"] * df["unit_price"]`
- Group: `df.groupby("category")["revenue"].sum().sort_values(ascending=False)`

In [150]:
# TODO: revenue by category using the RAW orders (delivered only)
revenue_raw = orders[orders["status"] == "delivered"]
revenue_raw["revenue"] = orders["quantity"] * orders["unit_price"]
revenue_raw.groupby("category")["revenue"].sum().sort_values(ascending=False)


category
Laptops       5197.98
Monitors      2190.00
Headphones     626.50
Keyboards      270.00
Name: revenue, dtype: float64

## 3. Find the problems (5 min)

**Hints**
- Duplicates: `df.duplicated().sum()`, and `df[df.duplicated(keep=False)]` to see them
- Missing values: `df.isna().sum()`
- Orphan rows: `~orders["customer_id"].isin(customers["customer_id"])`

In [151]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
from IPython.display import display


# TODO: 1. how many fully duplicated rows are there? Show them.
dup_tot=orders.duplicated()
dup=orders[orders.duplicated(keep=False)]
print('Duplicated rows:')
display(dup)

# TODO: 2. how many missing values in each column?
print('Missing values:',orders.isnull().sum().sum())
display(orders[orders.isnull().any(axis=1)])

# TODO: 3. which orders belong to a customer who is NOT in the customers table?
print('Orders with missing customer:',list(orders[~orders["customer_id"].isin(customers["customer_id"])].customer_id))
display(orders[~orders["customer_id"].isin(customers["customer_id"])])

Duplicated rows:


,order_id,customer_id,order_date,category,quantity,unit_price,status
17,118,8,2024-05-25,Headphones,2.0,89.5,delivered
20,118,8,2024-05-25,Headphones,2.0,89.5,delivered


Missing values: 2


,order_id,customer_id,order_date,category,quantity,unit_price,status
8,109,3,2024-05-12,Keyboards,NaN,45.0,delivered
9,110,7,2024-05-13,Headphones,1.0,NaN,delivered


Orders with missing customer: [9]


,order_id,customer_id,order_date,category,quantity,unit_price,status
21,121,9,2024-05-29,Keyboards,1.0,45.0,delivered


## 4. Clean it (7 min)

**Hints**
- `orders_clean = orders.drop_duplicates()`
- Fill a price with the median for that category:
  `df.groupby("category")["unit_price"].transform("median")`
- `df["quantity"].fillna(1)` — one unit is a reasonable default for this shop
- `pd.to_datetime(df["order_date"])`
- Make quantity an integer at the end: `.astype(int)`

In [152]:
orders_clean = orders.copy()

# TODO: 1. drop duplicate rows
orders_clean = orders.drop_duplicates()

# TODO: 2. fill missing unit_price with the median price of the same category
orders_clean['unit_price']=orders_clean['unit_price'].fillna(orders_clean.groupby('category')['unit_price'].transform('median'))

# TODO: 3. fill missing quantity with 1, then convert the column to int
orders_clean['quantity']=orders_clean['quantity'].fillna(1)

# TODO: 4. convert order_date to datetime
orders_clean['order_date']=pd.to_datetime(orders_clean['order_date'])

# TODO: 4A. make quantity an integer at the end
orders_clean['quantity']=orders_clean['quantity'].astype(int)

orders_clean.info()
print('\n')
print('='*30)
display(orders_clean)


<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 21
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     21 non-null     int64         
 1   customer_id  21 non-null     int64         
 2   order_date   21 non-null     datetime64[us]
 3   category     21 non-null     str           
 4   quantity     21 non-null     int64         
 5   unit_price   21 non-null     float64       
 6   status       21 non-null     str           
dtypes: datetime64[us](1), float64(1), int64(3), str(2)
memory usage: 1.3 KB




,order_id,customer_id,order_date,category,quantity,unit_price,status
0,101,1,2024-05-02,Laptops,1,1299.99,delivered
1,102,2,2024-05-02,Headphones,2,89.50,delivered
2,103,1,2024-05-04,Headphones,1,89.50,returned
3,104,3,2024-05-05,Monitors,2,219.00,delivered
4,105,4,2024-05-07,Laptops,1,1499.00,cancelled
5,106,5,2024-05-08,Keyboards,3,45.00,delivered
6,107,2,2024-05-09,Monitors,1,219.00,delivered
7,108,6,2024-05-11,Laptops,1,1099.00,delivered
8,109,3,2024-05-12,Keyboards,1,45.00,delivered
9,110,7,2024-05-13,Headphones,1,89.50,delivered


## 5. NumPy: revenue column (3 min)

Cancelled and returned orders earn **no** revenue.

**Hints**
- `np.where(condition, value_if_true, value_if_false)`
- Condition: `orders_clean["status"] == "delivered"`
- `np.round(x, 2)` if you want tidy numbers

In [153]:
# TODO: create a "revenue" column that is 0 for cancelled and returned orders
orders_clean["revenue"] = np.where(orders_clean['status']=='delivered',(orders_clean['quantity']*orders_clean['unit_price']),0)

revenue_clean=orders_clean

# TODO: print total revenue, the mean order value, and the largest single order
print(f'Total revenue: {orders_clean['revenue'][orders_clean['quantity']!=0].sum()}')
print(f'\nThe mean order value: {orders_clean['revenue'].mean()}')
print(f'\nThe largest single order: {orders_clean['revenue'].max()}')

Total revenue: 8239.98

The mean order value: 392.38

The largest single order: 1499.0


## 6. Merge and answer the business question (5 min)

**Hints**
- `orders_clean.merge(customers, on="customer_id", how="left")`
- A left merge keeps the order from the unknown customer, with `city` as NaN. That is
  usually what you want: do not silently delete money.
- `.groupby("city")["revenue"].sum().sort_values(ascending=False)`

In [154]:
# TODO: merge customers into orders_clean, then find revenue by city
merged = orders_clean.merge(customers,on='customer_id',how='left')

revenue_by_city = merged.groupby('city')['revenue'].sum()
print(revenue_by_city)

city
Calgary     1318.00
Montreal    1678.00
Ottawa      4227.98
Toronto      971.00
Name: revenue, dtype: float64


## 7. Compare with step 2 (2 min)

**Hint:** `pd.DataFrame({"raw": revenue_raw, "clean": revenue_clean})` lines the two up side by side.

In [ ]:
# TODO: revenue by category from the CLEANED data, then compare it with revenue_raw
revenue_clean = merged.groupby("category")["revenue"].sum().sort_values(ascending=False)

comparison = pd.DataFrame({'raw':revenue_raw,'clean':revenue_clean})
comparison

,raw,clean
category,,
Laptops,5197.98,5197.98
Monitors,2190.00,2190.00
Headphones,626.50,537.00
Keyboards,270.00,315.00


**Question:** two categories changed. Which ones, and what caused each change?

*Your answer:*

The categories that have changed are 'Headphones' and 'Keyboards'

The reason is because after cleaning process there weren't duplicated rows, no Nan values, and no orders with missing customers. Nan values were calculated according especific criteria



## 8. Wrap up (optional, if time)

Write two sentences for the NorthLine ops team: which category to push, and one data
quality issue they should fix at the source.

*Your answer:*
# 1. After getting groups by category, Laptops is the one to invest the most

# 2. Orders with wrong customers id could represent not only a wrong idea of what is happening but a huge financial loss 
---

### Submission
Save the notebook, push it to your GitHub repo, and paste the repo link in the class channel.

### Stretch goals
1. Which customer spent the most? (merge, then group by name)
2. Add a `week` column from `order_date` and show revenue per week.
   Hint: `df["order_date"].dt.isocalendar().week`
3. Ask Claude or Copilot to write the cleaning step for you, then compare it with yours.
   Did it make the same choices about the missing quantity? Was it right?

In [164]:
print('Customer with more spenses: ',merged.groupby('customer_id')['revenue'].sum().idxmax())
orders_clean['week']=orders_clean['order_date'].dt.isocalendar().week
print('\nRevenue per week')
display(orders_clean.groupby('week')['revenue'].sum())

Customer with more spenses:  3

Revenue per week


week
18    1916.99
19    1498.00
20    1145.00
21    3196.99
22     483.00
Name: revenue, dtype: float64

### Claude calculation

In [165]:
# Claude's approach - more concise with method chaining

# Step 1: Copy and drop duplicates
orders_clean_v2 = orders.drop_duplicates().copy()

# Step 2-4: Clean in one go using assign() for clarity
orders_clean_v2 = (orders_clean_v2
    .assign(
        unit_price=orders_clean_v2['unit_price'].fillna(
            orders_clean_v2.groupby('category')['unit_price'].transform('median')
        ),
        quantity=orders_clean_v2['quantity'].fillna(1),
        order_date=pd.to_datetime(orders_clean_v2['order_date'])
    )
    .astype({'quantity': 'int64'})
)

print("Claude's approach shape:", orders_clean_v2.shape)
display(orders_clean_v2.info())


Claude's approach shape: (21, 7)
<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 21
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     21 non-null     int64         
 1   customer_id  21 non-null     int64         
 2   order_date   21 non-null     datetime64[us]
 3   category     21 non-null     str           
 4   quantity     21 non-null     int64         
 5   unit_price   21 non-null     float64       
 6   status       21 non-null     str           
dtypes: datetime64[us](1), float64(1), int64(3), str(2)
memory usage: 1.3 KB


None

### Comparison

In [168]:
# COMPARISON: Your approach vs Claude's approach

# Reset indices
orders_clean_reset = orders_clean.reset_index(drop=True)
orders_clean_v2_reset = orders_clean_v2.reset_index(drop=True)

print("=" * 50)
print("COMPARISON: Your Cleaning vs Claude's Cleaning")
print("=" * 50)

# Compare only the shared columns (before revenue was added)
shared_cols = ['order_id', 'customer_id', 'order_date', 'category', 'quantity', 'unit_price', 'status']

# 1. Shape comparison
print(f"\n✓ Shape matches: {orders_clean_reset[shared_cols].shape == orders_clean_v2_reset[shared_cols].shape}")

# 2. Data type comparison
print(f"✓ Data types match: {orders_clean_reset[shared_cols].dtypes.equals(orders_clean_v2_reset[shared_cols].dtypes)}")

# 3. Values comparison
all_match = orders_clean_reset[shared_cols].equals(orders_clean_v2_reset[shared_cols])
print(f"✓ Data values match: {all_match}")

if all_match:
    print("\n✅ Both approaches produced IDENTICAL results!")
else:
    print("\n❌ Differences found - checking each column:")
    for col in shared_cols:
        match = orders_clean_reset[col].equals(orders_clean_v2_reset[col])
        print(f"  {col}: {'✓ Match' if match else '✗ Mismatch'}")

# 4. Summary statistics
print("\n✓ Summary Statistics:")
print(f"  Total rows: {len(orders_clean_reset)} == {len(orders_clean_v2_reset)}")
print(f"  Null values: {orders_clean_reset[shared_cols].isnull().sum().sum()} == {orders_clean_v2_reset[shared_cols].isnull().sum().sum()}")

# 5. Show sample rows
print("\nSample comparison (first 3 rows):")
print("\nYour approach:")
display(orders_clean_reset[shared_cols].head(3))
print("\nClaude's approach:")
display(orders_clean_v2_reset[shared_cols].head(3))


COMPARISON: Your Cleaning vs Claude's Cleaning

✓ Shape matches: True
✓ Data types match: True
✓ Data values match: True

✅ Both approaches produced IDENTICAL results!

✓ Summary Statistics:
  Total rows: 21 == 21
  Null values: 0 == 0

Sample comparison (first 3 rows):

Your approach:


,order_id,customer_id,order_date,category,quantity,unit_price,status
0,101,1,2024-05-02,Laptops,1,1299.99,delivered
1,102,2,2024-05-02,Headphones,2,89.50,delivered
2,103,1,2024-05-04,Headphones,1,89.50,returned



Claude's approach:


,order_id,customer_id,order_date,category,quantity,unit_price,status
0,101,1,2024-05-02,Laptops,1,1299.99,delivered
1,102,2,2024-05-02,Headphones,2,89.50,delivered
2,103,1,2024-05-04,Headphones,1,89.50,returned
